# AI/BI dashboards, sharing and alerts

AI/BI dashboards (formerly Lakeview dashboards) turn SQL datasets into shareable, interactive reports. You build datasets on the Data tab, add visualizations, filters and parameters on the canvas, then publish. Publishing chooses whose permissions viewers query with, either the publisher's shared data permissions or each viewer's own (individual data permissions, which enforce row-level security per viewer). Schedules email or post snapshots to subscribers. SQL alerts run a query on a schedule and notify when a condition is met.

**How to run:** attach **Serverless** compute (top right), then run the cells from top to bottom. Each part asks you to predict the result first; the expected results are listed at the end.

From the Learning Hub lesson: [AI/BI dashboards, sharing and alerts](https://www.datapreplab.com/hub#topic/aibi-dashboards) · Independent study resource, not affiliated with or endorsed by Databricks.

**Goal:** build a small dashboard with a parameter, publish it and create an alert.

**Part A: data.**

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.default.orders_dash_lab AS
SELECT id AS order_id,
       element_at(array('EU', 'US', 'APAC'), CAST(id % 3 + 1 AS INT)) AS region,
       date_sub(current_date(), CAST(id % 14 AS INT)) AS order_date,
       CAST(20 + id % 80 AS DECIMAL(10,2)) AS amount
FROM range(600);

**Part B: dataset and visuals.** Create a dashboard (**New > Dashboard**). On the Data tab, add this dataset:

```text
SELECT order_date, region, count(*) AS orders, sum(amount) AS revenue
FROM workspace.default.orders_dash_lab
WHERE (:dates.min IS NULL OR order_date >= :dates.min)
  AND (:dates.max IS NULL OR order_date <= :dates.max)
GROUP BY ALL
```

On the canvas, add a bar chart of revenue by region, a line chart of orders by order_date, and a date range filter widget bound to the `dates` parameter.

**Part C: publish.** Predict who sees what with each option, then publish with shared data permissions and open the published view. Change a chart in the draft and confirm the published view does not change until you publish again. Then click **Schedule** on the published dashboard to see the schedule options; cancel without creating one, or create one and delete it at cleanup.

**Part D: an alert.** Create an alert (**Alerts > Create alert**) with this query, condition `orders < 100`, and a daily schedule:

```text
SELECT count(*) AS orders FROM workspace.default.orders_dash_lab
WHERE order_date = date_sub(current_date(), 1)
```

In the condition, choose **First row**, column `orders`, operator `<` and **Static value** `100`, then click **Test condition**. Under **Notifications**, add yourself and set **When alerting, notify** to **Once** until the status is OK.

Expected evidence:

```text
Part B -> the date range parameter appears with a default range ; 3 regions, 200 orders each over 14 days, about 14 per region per day ; revenue EU 11,620, US 11,660, APAC 11,620
Part C -> published view unchanged until you publish again ; Schedule on the published dashboard opens Add schedule (every day, week or a cron, a time zone, advanced settings for subscribers)
Part D -> Test condition shows Triggered with orders 43 (43 < 100)
```

**Cleanup:** delete the dashboard and alert, then `DROP TABLE workspace.default.orders_dash_lab;`